## Local-Vision Transformer model implementation

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------------
# Local MLP Module with Convolution
# -------------------------------
class LocalMLP(nn.Module):
    """
    A modified MLP that injects locality by inserting a depthwise convolution
    between two linear layers. The input is first linearly projected, then reshaped
    to spatial dimensions to apply a 3x3 depthwise conv, and then projected back.
    """
    def __init__(self, in_channels, hidden_channels, dropout=0.):
        super().__init__()
        self.fc1 = nn.Linear(in_channels, hidden_channels)
        self.drop1 = nn.Dropout(dropout)
        # Depthwise convolution: groups equals the number of channels
        self.dwconv = nn.Conv2d(hidden_channels, hidden_channels, kernel_size=3,
                                padding=1, groups=hidden_channels)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_channels, in_channels)
        self.drop2 = nn.Dropout(dropout)
        self.hidden_channels = hidden_channels

    def forward(self, x):
        # x: (B, C, H, W)
        B, C, H, W = x.shape
        # Flatten spatial dimensions for the linear layer: (B, N, C) with N=H*W
        x = x.flatten(2).transpose(1, 2)  # shape: (B, N, C)
        x = self.fc1(x)
        x = self.drop1(x)
        # Reshape to (B, hidden_channels, H, W) for convolution
        x = x.transpose(1, 2).view(B, self.hidden_channels, H, W)
        x = self.dwconv(x)
        x = self.act(x)
        # Flatten back to (B, N, hidden_channels)
        x = x.flatten(2).transpose(1, 2)
        x = self.fc2(x)
        x = self.drop2(x)
        return x

# -------------------------------
# LocalViT Transformer Block
# -------------------------------
class LocalViTBlock(nn.Module):
    """
    A transformer block that incorporates global self-attention and a local MLP branch.
    The local branch first reshapes the normalized token sequence into a 2D feature map,
    applies the local (convolution-based) MLP, and then re-flattens back.
    """
    def __init__(self, dim, num_heads, mlp_ratio=4., dropout=0.):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, dropout=dropout)
        self.norm2 = nn.LayerNorm(dim)
        hidden_dim = int(dim * mlp_ratio)
        self.mlp = LocalMLP(in_channels=dim, hidden_channels=hidden_dim, dropout=dropout)

    def forward(self, x, H, W):
        """
        Args:
            x: Token sequence of shape (B, N, dim) where N = H * W.
            H, W: Spatial dimensions corresponding to the tokens.
        """
        # Global self-attention branch
        x_norm = self.norm1(x)
        # nn.MultiheadAttention expects (S, B, dim)
        attn_out, _ = self.attn(x_norm.transpose(0, 1),
                                  x_norm.transpose(0, 1),
                                  x_norm.transpose(0, 1))
        x = x + attn_out.transpose(0, 1)

        # Local MLP branch with convolution
        x_norm = self.norm2(x)
        B, N, C = x_norm.shape
        # Reshape tokens to 2D feature map (B, C, H, W)
        x_2d = x_norm.transpose(1, 2).view(B, C, H, W)
        x_mlp = self.mlp(x_2d)
        # x_mlp is (B, N, C) after LocalMLP
        x = x + x_mlp
        return x

# -------------------------------
# Overall LocalViT Model
# -------------------------------
class LocalViT(nn.Module):
    """
    The overall model that uses a convolution-based patch embedding followed by a series
    of LocalViTBlocks. A learnable positional embedding is added to the flattened patch tokens.
    Finally, global average pooling and a classifier head produce the output.

    This implementation is based on “LocalViT: Analyzing Locality in Vision Transformers.”
    """
    def __init__(self, img_size=224, in_channels=3, num_classes=1000, embed_dim=64,
                 depth=12, num_heads=4, mlp_ratio=4., dropout=0.):
        super().__init__()
        # Patch embedding: a conv layer to produce patch tokens from the image.
        self.patch_embed = nn.Conv2d(in_channels, embed_dim, kernel_size=7, stride=4, padding=3)
        # Compute spatial dimensions after patch embedding (assuming square input)
        self.H = img_size // 4
        self.W = img_size // 4
        num_patches = self.H * self.W

        # Learnable positional embedding added to patch tokens.
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
        self.pos_drop = nn.Dropout(p=dropout)

        # Stack of LocalViT transformer blocks
        self.blocks = nn.ModuleList([
            LocalViTBlock(dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, dropout=dropout)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        # Classification head
        self.head = nn.Linear(embed_dim, num_classes)

        # Initialize positional embedding
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        """
        Args:
            x: Input image tensor of shape (B, in_channels, img_size, img_size)
        """
        x = self.patch_embed(x)  # shape: (B, embed_dim, H, W)
        B, C, H, W = x.shape
        # Flatten spatial dimensions to tokens: (B, N, embed_dim)
        x = x.flatten(2).transpose(1, 2)
        x = x + self.pos_embed
        x = self.pos_drop(x)

        # Process through LocalViT blocks
        for block in self.blocks:
            x = block(x, H, W)
        x = self.norm(x)
        # Global average pooling over tokens
        x = x.mean(dim=1)
        x = self.head(x)
        return x

# -------------------------------
# Testing the LocalViT Model
# -------------------------------
if __name__ == '__main__':
    # Create a LocalViT instance
    model = LocalViT(img_size=224, in_channels=3, num_classes=10, embed_dim=64,
                     depth=8, num_heads=4, mlp_ratio=4., dropout=0.1)
    # Dummy input: batch of 2 images with 3 channels and 224x224 resolution
    x = torch.randn(2, 3, 224, 224)
    logits = model(x)
    print("Logits shape:", logits.shape)  # Expected output: (2, 10)


Logits shape: torch.Size([2, 10])
